# RoBERTa Masked Language Modelling (TAPT)

This notebook performs **task-adaptive pretraining (TAPT)** for `FacebookAI/roberta-large` on the reappraisal corpus. It is designed as a controlled precursor to the existing soft-label classification experiment.

The evaluation safeguards are deliberate:

- the downstream train/validation/test split is created **before** MLM;
- MLM sees only the downstream training fold;
- a small internal portion of that training fold is used to select the MLM epoch;
- validation and test texts never enter MLM training or MLM model selection;
- a fresh adapted checkpoint is required for every random seed or held-out situation.

The primary TAPT corpus contains only `REAPPRAISAL`. The six situation texts repeat many times and could otherwise dominate this small corpus. Set `INCLUDE_SITUATION = True` for a planned secondary ablation.

In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ModuleNotFoundError:
    MultilabelStratifiedShuffleSplit = None
    HAS_ITERSTRAT = False

print('PyTorch:', torch.__version__)
print('Iterative stratification available:', HAS_ITERSTRAT)

## Configuration

The default configuration does not start training. For a quick end-to-end check, set `RUN_TAPT = True` and leave `SMOKE_TEST = True`. For the actual experiment, also set `SMOKE_TEST = False`.

In [ ]:
def find_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'data' / 'annotate-lime.csv').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate data/annotate-lime.csv; add the private data file and start Jupyter from inside the repository.'
    )


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'
BASE_OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'tapt' / 'masked_language_modeling'

MODEL_NAME = 'FacebookAI/roberta-large'
SEED = 13
SPLIT_MODE = 'random'              # 'random' or 'leave_one_vignette'
HELD_OUT_SITUATION_INDEX = 0       # used only for leave_one_vignette
INCLUDE_SITUATION = False          # recommended primary analysis
LOCAL_FILES_ONLY = False

MAX_LENGTH = 256
MLM_PROBABILITY = 0.15
BATCH_SIZE = 1                     # conservative for RoBERTa-large
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
PATIENCE = 2
MLM_VALIDATION_FRACTION = 0.10

# Safe defaults: Run All does not train; enabling training starts with the smoke test.
RUN_TAPT = False
SMOKE_TEST = True                # 8 texts, one epoch
SKIP_COMPLETED_TAPT = True
ALLOW_TAPT_OVERWRITE = False
SMOKE_TEST_ITEMS = 8

REAPPRAISAL_TECHNIQUES = ['[GM]', '[PR]', '[A]', '[EJ]', '[N]', '[UC]', '[V]', '[H]']
LABEL_COLS = [tech.strip('[]') for tech in REAPPRAISAL_TECHNIQUES]

print('Project directory:', PROJECT_DIR)
print('Base model:', MODEL_NAME)
print('Run enabled:', RUN_TAPT, '| smoke test:', SMOKE_TEST)

## Reproducibility and device

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


set_seed(SEED)
device = get_device()
print('Using device:', device)
if device.type == 'cpu' and MODEL_NAME == 'FacebookAI/roberta-large':
    print(
        'Warning: RoBERTa-large MLM on CPU will be slow. Run the 8-item smoke test first, '
        'then use a CUDA/MPS runtime for the full experiment if one is available.'
    )

## Load the corpus and reproduce the classification split

The annotation matrices are loaded only to reproduce the same multilabel-stratified split as the classifier. The MLM objective never uses the labels.

In [ ]:
def load_reappraisal_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    texts = pd.read_csv(data_dir / 'annotate-lime.csv')
    soft_labels = pd.DataFrame()

    for technique in REAPPRAISAL_TECHNIQUES:
        label = technique.strip('[]')
        annotations = pd.read_csv(
            data_dir / f'annotations-only-{technique}.csv',
            sep=';',
        ).apply(pd.to_numeric, errors='coerce')
        soft_labels[label] = annotations.mean(axis=0).values

    result = pd.concat(
        [
            texts[['key', 'SITUATION', 'REAPPRAISAL']].reset_index(drop=True),
            soft_labels.reset_index(drop=True),
        ],
        axis=1,
    )
    result['vignette_id'] = pd.factorize(result['SITUATION'])[0]
    return result


def multilabel_split_indices(
    frame: pd.DataFrame,
    binary_targets: np.ndarray,
    test_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    if HAS_ITERSTRAT:
        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed,
        )
        return next(splitter.split(frame, binary_targets))

    print('Warning: using a seeded random split; iterative-stratification is unavailable.')
    indices = np.arange(len(frame))
    return train_test_split(
        indices, test_size=test_size, random_state=seed, shuffle=True
    )


def make_random_split(
    frame: pd.DataFrame,
    seed: int,
    test_size: float = 0.15,
    val_size: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    majority = (frame[LABEL_COLS].to_numpy() > 0.50).astype(int)
    train_indices, temporary_indices = multilabel_split_indices(
        frame, majority, test_size=test_size + val_size, seed=seed
    )
    train_frame = frame.iloc[train_indices].reset_index(drop=True)
    temporary = frame.iloc[temporary_indices].reset_index(drop=True)
    temporary_majority = (temporary[LABEL_COLS].to_numpy() > 0.50).astype(int)
    val_indices, test_indices = multilabel_split_indices(
        temporary,
        temporary_majority,
        test_size=test_size / (test_size + val_size),
        seed=seed,
    )
    return (
        train_frame,
        temporary.iloc[val_indices].reset_index(drop=True),
        temporary.iloc[test_indices].reset_index(drop=True),
    )


def make_leave_one_vignette_split(
    frame: pd.DataFrame,
    seed: int,
    held_out_index: int,
    val_size: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    situations = list(frame['SITUATION'].drop_duplicates())
    held_out_situation = situations[held_out_index]
    train_val = frame.loc[frame['SITUATION'] != held_out_situation].reset_index(drop=True)
    test_frame = frame.loc[frame['SITUATION'] == held_out_situation].reset_index(drop=True)
    majority = (train_val[LABEL_COLS].to_numpy() > 0.50).astype(int)
    train_indices, val_indices = multilabel_split_indices(
        train_val, majority, test_size=val_size, seed=seed
    )
    return (
        train_val.iloc[train_indices].reset_index(drop=True),
        train_val.iloc[val_indices].reset_index(drop=True),
        test_frame,
        held_out_situation,
    )

In [ ]:
dataset = load_reappraisal_dataset()

if SPLIT_MODE == 'random':
    train_df, val_df, test_df = make_random_split(dataset, SEED)
    split_label = f'random_seed{SEED}'
    held_out_situation = None
elif SPLIT_MODE == 'leave_one_vignette':
    train_df, val_df, test_df, held_out_situation = make_leave_one_vignette_split(
        dataset, SEED, HELD_OUT_SITUATION_INDEX
    )
    split_label = f'leave_vignette_{HELD_OUT_SITUATION_INDEX}_seed{SEED}'
else:
    raise ValueError(f'Unknown SPLIT_MODE: {SPLIT_MODE}')

train_keys = set(train_df['key'])
val_keys = set(val_df['key'])
test_keys = set(test_df['key'])
assert train_keys.isdisjoint(val_keys)
assert train_keys.isdisjoint(test_keys)
assert val_keys.isdisjoint(test_keys)

print('Dataset size:', len(dataset))
print('Train / validation / test:', len(train_df), len(val_df), len(test_df))
print('Split label:', split_label)
if held_out_situation is not None:
    print('Held-out situation:', held_out_situation)
display(train_df[['key', 'SITUATION', 'REAPPRAISAL']].head())

## MLM dataset and dynamic masking

`DataCollatorForLanguageModeling` creates new masks dynamically. `AtLeastOneMaskCollator` additionally guarantees that every example contains a prediction target, preventing undefined MLM losses when the physical batch size is one. Special tokens and padding tokens are excluded automatically. Validation masking is recreated with the same seed each epoch so validation losses are comparable.

In [ ]:
class ReappraisalMLMDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        max_length: int,
        include_situation: bool = False,
    ):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_situation = include_situation

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        row = self.frame.iloc[index]
        if self.include_situation:
            return self.tokenizer(
                str(row['SITUATION']),
                str(row['REAPPRAISAL']),
                truncation=True,
                max_length=self.max_length,
                add_special_tokens=True,
            )
        return self.tokenizer(
            str(row['REAPPRAISAL']),
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )


class AtLeastOneMaskCollator:
    def __init__(self, tokenizer, mlm_probability: float, seed: int):
        self.tokenizer = tokenizer
        self.base_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=True,
            mlm_probability=mlm_probability,
            return_tensors='pt',
            seed=seed,
        )
        self.generator = torch.Generator().manual_seed(seed)

    def __call__(self, features):
        batch = self.base_collator(features)

        for row in range(batch['labels'].shape[0]):
            if (batch['labels'][row] != -100).any():
                continue

            input_ids = batch['input_ids'][row]
            valid_positions = batch['attention_mask'][row].bool()
            for special_id in self.tokenizer.all_special_ids:
                valid_positions &= input_ids != special_id

            candidates = torch.where(valid_positions)[0]
            if len(candidates) == 0:
                raise RuntimeError('Example contains no maskable tokens.')

            choice = torch.randint(
                len(candidates),
                size=(1,),
                generator=self.generator,
            ).item()
            position = candidates[choice]
            batch['labels'][row, position] = input_ids[position]
            batch['input_ids'][row, position] = self.tokenizer.mask_token_id

        return batch


def split_mlm_train_validation(
    downstream_train: pd.DataFrame,
    validation_fraction: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    shuffled = downstream_train.sample(frac=1, random_state=seed).reset_index(drop=True)
    validation_size = max(1, round(len(shuffled) * validation_fraction))
    mlm_validation = shuffled.iloc[:validation_size].reset_index(drop=True)
    mlm_train = shuffled.iloc[validation_size:].reset_index(drop=True)
    assert set(mlm_train['key']).isdisjoint(set(mlm_validation['key']))
    assert set(mlm_train['key']) | set(mlm_validation['key']) == set(downstream_train['key'])
    return mlm_train, mlm_validation

## TAPT training loop

In [ ]:
@dataclass(frozen=True)
class TAPTConfig:
    model_name: str
    seed: int
    max_length: int
    mlm_probability: float
    batch_size: int
    gradient_accumulation_steps: int
    learning_rate: float
    weight_decay: float
    num_epochs: int
    patience: int
    validation_fraction: float
    include_situation: bool
    local_files_only: bool
    downstream_run_id: str


def evaluate_mlm(
    model,
    validation_dataset: Dataset,
    tokenizer,
    config: TAPTConfig,
    device: torch.device,
) -> float:
    fixed_validation_collator = AtLeastOneMaskCollator(
        tokenizer=tokenizer,
        mlm_probability=config.mlm_probability,
        seed=config.seed,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=fixed_validation_collator,
    )

    model.eval()
    loss_sum = 0.0
    masked_token_count = 0
    with torch.no_grad():
        for step, batch in enumerate(validation_loader, start=1):
            batch = {name: tensor.to(device) for name, tensor in batch.items()}
            batch_masked_tokens = int((batch['labels'] != -100).sum().item())
            if batch_masked_tokens == 0:
                raise RuntimeError(f'Validation batch {step} has no prediction targets.')
            validation_loss = model(**batch).loss
            if not torch.isfinite(validation_loss):
                raise RuntimeError(f'Non-finite validation loss at step {step}.')
            loss_sum += validation_loss.item() * batch_masked_tokens
            masked_token_count += batch_masked_tokens
    return loss_sum / masked_token_count


def run_tapt(
    downstream_train: pd.DataFrame,
    output_dir: Path,
    config: TAPTConfig,
    device: torch.device,
) -> pd.DataFrame:
    set_seed(config.seed)
    output_dir.mkdir(parents=True, exist_ok=True)

    mlm_train_df, mlm_validation_df = split_mlm_train_validation(
        downstream_train, config.validation_fraction, config.seed
    )
    print('MLM train / internal validation:', len(mlm_train_df), len(mlm_validation_df))

    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name, local_files_only=config.local_files_only
    )
    model = AutoModelForMaskedLM.from_pretrained(
        config.model_name, local_files_only=config.local_files_only
    ).to(device)

    mlm_train_dataset = ReappraisalMLMDataset(
        mlm_train_df, tokenizer, config.max_length, config.include_situation
    )
    mlm_validation_dataset = ReappraisalMLMDataset(
        mlm_validation_df, tokenizer, config.max_length, config.include_situation
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    batches_per_epoch = math.ceil(len(mlm_train_dataset) / config.batch_size)
    updates_per_epoch = math.ceil(batches_per_epoch / config.gradient_accumulation_steps)
    total_updates = updates_per_epoch * config.num_epochs
    warmup_updates = round(0.10 * total_updates)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_updates,
        num_training_steps=total_updates,
    )

    history = []
    best_validation_loss = float('inf')
    best_epoch = None
    bad_epochs = 0

    for epoch in range(1, config.num_epochs + 1):
        dynamic_train_collator = AtLeastOneMaskCollator(
            tokenizer=tokenizer,
            mlm_probability=config.mlm_probability,
            seed=config.seed + epoch,
        )
        generator = torch.Generator().manual_seed(config.seed + epoch)
        train_loader = DataLoader(
            mlm_train_dataset,
            batch_size=config.batch_size,
            shuffle=True,
            generator=generator,
            collate_fn=dynamic_train_collator,
        )

        model.train()
        optimizer.zero_grad()
        train_loss_sum = 0.0
        train_masked_token_count = 0
        accumulated_masked_token_count = 0

        for step, batch in enumerate(train_loader, start=1):
            batch = {name: tensor.to(device) for name, tensor in batch.items()}
            batch_masked_tokens = int((batch['labels'] != -100).sum().item())
            if batch_masked_tokens == 0:
                raise RuntimeError(
                    f'Training batch at epoch {epoch}, step {step} has no prediction targets.'
                )
            raw_loss = model(**batch).loss
            if not torch.isfinite(raw_loss):
                raise RuntimeError(
                    f'Non-finite training loss at epoch {epoch}, step {step}.'
                )
            train_loss_sum += raw_loss.item() * batch_masked_tokens
            train_masked_token_count += batch_masked_tokens
            accumulated_masked_token_count += batch_masked_tokens
            (raw_loss * batch_masked_tokens).backward()

            should_update = (
                step % config.gradient_accumulation_steps == 0
                or step == len(train_loader)
            )
            if should_update:
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.div_(accumulated_masked_token_count)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                accumulated_masked_token_count = 0

        train_loss = train_loss_sum / train_masked_token_count
        validation_loss = evaluate_mlm(
            model, mlm_validation_dataset, tokenizer, config, device
        )
        if not np.isfinite(validation_loss):
            raise RuntimeError(f'Non-finite validation loss at epoch {epoch}.')
        row = {
            'epoch': epoch,
            'train_mlm_loss': train_loss,
            'validation_mlm_loss': validation_loss,
            'validation_perplexity': math.exp(min(validation_loss, 20)),
            'learning_rate': scheduler.get_last_lr()[0],
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d} | train MLM loss={train_loss:.4f} | "
            f"validation MLM loss={validation_loss:.4f} | "
            f"perplexity={row['validation_perplexity']:.2f}"
        )

        if validation_loss < best_validation_loss - 1e-4:
            best_validation_loss = validation_loss
            best_epoch = epoch
            bad_epochs = 0
            model.base_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
        else:
            bad_epochs += 1

        pd.DataFrame(history).to_csv(output_dir / 'mlm_training_history.csv', index=False)
        if bad_epochs >= config.patience:
            print(f'Early stopping after epoch {epoch}; best epoch was {best_epoch}.')
            break

    manifest = {
        'config': asdict(config),
        'downstream_run_id': config.downstream_run_id,
        'best_epoch': best_epoch,
        'best_validation_mlm_loss': best_validation_loss,
        'downstream_train_items': len(downstream_train),
        'mlm_train_items': len(mlm_train_df),
        'mlm_validation_items': len(mlm_validation_df),
        'adapted_encoder_directory': str(output_dir),
    }
    (output_dir / 'tapt_manifest.json').write_text(json.dumps(manifest, indent=2))

    assignments = pd.concat(
        [
            mlm_train_df[['key']].assign(mlm_assignment='train'),
            mlm_validation_df[['key']].assign(mlm_assignment='validation'),
        ],
        ignore_index=True,
    )
    assignments.to_csv(output_dir / 'mlm_split_assignments.csv', index=False)
    return pd.DataFrame(history)

## Launch the experiment

Smoke-test outputs are kept in a separate folder and should not be used in the final classification comparison.

In [ ]:
run_name = f'roberta_large_tapt_{split_label}'
training_frame = train_df.copy()
effective_epochs = NUM_EPOCHS

if SMOKE_TEST:
    run_name += '_smoke_test'
    training_frame = training_frame.sample(
        n=min(SMOKE_TEST_ITEMS, len(training_frame)),
        random_state=SEED,
    ).reset_index(drop=True)
    effective_epochs = 1

ADAPTED_ENCODER_DIR = BASE_OUTPUT_DIR / run_name

tapt_config = TAPTConfig(
    model_name=MODEL_NAME,
    seed=SEED,
    max_length=MAX_LENGTH,
    mlm_probability=MLM_PROBABILITY,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=effective_epochs,
    patience=PATIENCE,
    validation_fraction=MLM_VALIDATION_FRACTION,
    include_situation=INCLUDE_SITUATION,
    local_files_only=LOCAL_FILES_ONLY,
    downstream_run_id=split_label,
)

print('Output directory:', ADAPTED_ENCODER_DIR)
print('Texts available to MLM:', len(training_frame))
display(pd.Series(asdict(tapt_config), name='value').to_frame())

history_path = ADAPTED_ENCODER_DIR / 'mlm_training_history.csv'
manifest_path = ADAPTED_ENCODER_DIR / 'tapt_manifest.json'
checkpoint_paths = [ADAPTED_ENCODER_DIR / 'model.safetensors', ADAPTED_ENCODER_DIR / 'pytorch_model.bin']
completed_tapt_run = manifest_path.exists() and any(path.exists() for path in checkpoint_paths)
has_existing_artifacts = ADAPTED_ENCODER_DIR.exists() and any(ADAPTED_ENCODER_DIR.iterdir())

if RUN_TAPT and completed_tapt_run and SKIP_COMPLETED_TAPT:
    history_df = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
    print('Skipping completed TAPT run:', ADAPTED_ENCODER_DIR)
elif RUN_TAPT:
    if has_existing_artifacts and not ALLOW_TAPT_OVERWRITE:
        raise FileExistsError(
            f'Refusing to overwrite existing TAPT artifacts in {ADAPTED_ENCODER_DIR}. '
            'Use a new run name or set ALLOW_TAPT_OVERWRITE = True deliberately.'
        )
    history_df = run_tapt(
        downstream_train=training_frame,
        output_dir=ADAPTED_ENCODER_DIR,
        config=tapt_config,
        device=device,
    )
else:
    history_df = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
    print('Training is disabled. Set RUN_TAPT = True in the configuration cell to opt in.')
    if not history_df.empty:
        print('Loaded existing training history:', history_path)

## Training diagnostics

MLM loss is a diagnostic, not the primary experimental outcome. The main result is whether this adapted encoder improves held-out soft-label classification relative to unadapted RoBERTa-large.

In [ ]:
if history_df.empty:
    print('No training history yet.')
else:
    display(history_df)
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.plot(history_df['epoch'], history_df['train_mlm_loss'], marker='o', label='Train')
    axis.plot(
        history_df['epoch'],
        history_df['validation_mlm_loss'],
        marker='o',
        label='Internal validation',
    )
    axis.set(
        xlabel='Epoch',
        ylabel='Masked-language-model loss',
        title='RoBERTa task-adaptive pretraining',
    )
    axis.legend()
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(ADAPTED_ENCODER_DIR / 'mlm_loss_curve.png', dpi=180, bbox_inches='tight')
    plt.show()

## Use the adapted encoder in the soft-label classifier

After a **full** run (`SMOKE_TEST = False`), use `ADAPTED_ENCODER_DIR` as the `model_name` in `BERT_Soft_Label_LR_Epoch_Tuning.ipynb`, set `local_files_only=True`, and keep every downstream hyperparameter identical to the unadapted RoBERTa-large run. The classifier validates `tapt_manifest.json` and refuses to run if the checkpoint's seed or held-out split does not exactly match its `run_id`.

```python
ROBERTA_LARGE_TAPT_CONFIG = replace(
    ROBERTA_LARGE_CONFIG,
    experiment_name='roberta_large_tapt_cls',
    model_name=str(ADAPTED_ENCODER_DIR),
    local_files_only=True,
)
```

Generate the baseline and TAPT checkpoint separately for each seed. For leave-one-vignette-out evaluation, generate a checkpoint separately for every held-out situation.

## Experiment summary and interpretation

The full seed-13 TAPT run adapted `FacebookAI/roberta-large` to the language of the reappraisal responses using masked-language modelling. Only the 424 examples in the downstream training partition were eligible for TAPT, preventing validation- or test-set leakage. These were divided into 382 MLM-training and 42 internal MLM-validation examples. Approximately 15% of eligible tokens were masked, and the model learned to predict them.

The corrected masking collator guaranteed at least one prediction target in every batch, so this full run contained no non-finite losses. Validation MLM loss was lowest at epoch 2 (`0.9865`, perplexity `2.6819`). It then worsened for two epochs, and early stopping correctly stopped training at epoch 4 and restored the epoch-2 checkpoint.

This result shows that the language-adaptation stage ran successfully; MLM loss is not a classification score. Whether the adapted representation is useful must be judged by a controlled downstream comparison with unadapted RoBERTa-large. That comparison found no clear overall benefit from TAPT for seed 13: TAPT slightly improved validation loss, RMSE, Spearman correlation, and majority macro-F1, while standard RoBERTa had slightly better MAE, Pearson correlation, and weak/strong-presence macro-F1. Because only one seed was evaluated, the result should be described as inconclusive rather than evidence that TAPT generally helps or harms performance.